# 🧠 Resume Intelligence Platform
### ML-Powered Resume ↔ Job Description Matching Engine

**Tech Stack:** Python | Gradio | scikit-learn | TF-IDF | NLTK  
**Domain:** Tech / Software / Data Science  
**Models:** Logistic Regression | Naive Bayes | SVM | Random Forest  
**Evaluation:** StratifiedKFold Cross-Validation (cv=5)  

---

### 📋 Architecture Overview
```
Resume (PDF/DOCX)
      │
      ▼
┌─────────────┐    ┌──────────────────┐
│ File Parser │    │ Job Description  │
│ (PDF/DOCX)  │    │ (Text Input)     │
└──────┬──────┘    └────────┬─────────┘
       │                    │
       ▼                    ▼
┌────────────────────────────────────┐
│        Text Preprocessor           │
│  lowercase → punctuation removal   │
│  → stopword removal → lemmatize    │
└──────────────────┬─────────────────┘
                   │
       ┌───────────┴───────────┐
       ▼                       ▼
┌─────────────┐       ┌─────────────────┐
│  TF-IDF     │       │  Skill Extractor │
│  Vectorizer │       │  (synonym map)   │
└──────┬──────┘       └────────┬─────────┘
       │                       │
       ▼                       ▼
┌──────────────┐      ┌──────────────────┐
│  ML Models   │      │  Skill Analysis  │
│  LR|NB|SVM   │      │  Matched/Missing │
│  |RF (cv=5)  │      │  /Extra Skills   │
└──────┬───────┘      └────────┬─────────┘
       │                       │
       ▼                       │
┌──────────────────────────────────────┐
│         Combined Scoring             │
│  0.6 × ML Proba + 0.4 × Cosine Sim  │
└──────────────────┬───────────────────┘
                   ▼
         Final Match Score (%)
         + Suggestions + Charts
```

## ⚙️ Step 1 — Install Dependencies

In [ ]:
# Install all required packages
!pip install gradio scikit-learn nltk PyPDF2 python-docx matplotlib seaborn datasets tabulate -q
print("✅ All packages installed successfully!")

## 📦 Step 2 — Imports & NLTK Setup

In [ ]:
import os
import re
import json
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import gradio as gr

# File parsing
import PyPDF2
import docx

# NLP & ML
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline
from sklearn.metrics.pairwise import cosine_similarity

try:
    from datasets import load_dataset
    HF_AVAILABLE = True
except ImportError:
    HF_AVAILABLE = False

warnings.filterwarnings('ignore')

# NLTK downloads
for resource in ['stopwords', 'wordnet', 'omw-1.4', 'punkt']:
    nltk.download(resource, quiet=True)

print('✅ All imports successful!')

## 🗂️ Step 3 — Skill Knowledge Base & Constants

In [ ]:
# ─────────────────────────────────────────────
# Comprehensive Tech / Software / DS skill taxonomy
# Each canonical skill maps to its synonyms/aliases
# ─────────────────────────────────────────────
SKILL_TAXONOMY = {
    # Programming Languages
    'python': ['py', 'python3', 'python2'],
    'javascript': ['js', 'ecmascript', 'es6', 'es2015'],
    'typescript': ['ts ', ' ts'],  # FIX: 'ts' hits 'projects','tests','its' — kept with spaces as guards
    'java': ['java8', 'java11', 'java17', 'jvm'],
    'c++': ['cpp', 'c plus plus'],
    'c#': ['csharp', 'c sharp', 'dotnet', '.net'],
    'r': ['r language', 'r programming', ' r,', ' r.', '(r)'],  # FIX: 'r' alone hits every word with r; use phrase guards
    'scala': ['scala lang'],
    'go': ['golang', 'go lang'],  # FIX: 'go' substring hits Django/MongoDB/algorithm — use full words for skill match
    'rust': ['rust lang'],
    'kotlin': [],
    'swift': [],
    'php': [],
    'ruby': ['ruby on rails', 'rails'],
    'sql': ['structured query language', 'mysql', 'postgresql', 'sqlite', 'mssql', 't-sql'],
    # ML / AI / DS
    # FIX: Removed single/double-char abbreviations that cause false positives.
    # 'ml' matches xml/html/yml; 'cv' matches "CV attached" (resume); 
    # 'rl' matches url/world; 'dl' acceptable (rare collision).
    # Full phrases are kept as primary matches; only safe aliases retained.
    'machine learning': ['statistical learning'],
    'deep learning': ['neural networks', 'ann', 'dnn'],
    'natural language processing': ['nlp', 'text mining', 'computational linguistics'],
    'computer vision': ['image processing', 'object detection'],
    'reinforcement learning': ['reward learning'],
    'data science': ['data analytics', 'analytics'],
    'data engineering': ['data pipeline', 'etl', 'data warehousing'],
    'feature engineering': ['feature extraction', 'feature selection'],
    'model deployment': ['mlops', 'model serving', 'productionization'],
    'generative ai': ['genai', 'llm', 'large language models', 'gpt', 'chatgpt'],
    # ML Frameworks
    'tensorflow': ['keras'],  # FIX: 'tf' is low-risk but keras is a valid distinct alias; tf kept as direct canonical match
    'pytorch': ['torch'],
    'scikit-learn': ['sklearn', 'scikit learn'],
    'xgboost': ['xgb', 'extreme gradient boosting'],
    'lightgbm': ['lgbm'],
    'hugging face': ['transformers', 'huggingface'],
    'opencv': ['cv2'],
    'numpy': ['numerical python'],  # FIX: 'np' hits 'input','nonprofit' — removed unsafe alias
    'pandas': ['dataframe', 'data wrangling'],
    'matplotlib': ['pyplot', 'plotting'],
    'seaborn': [],
    'plotly': [],
    # Cloud Platforms
    'aws': ['amazon web services', 'ec2', 's3', 'lambda', 'sagemaker'],
    'azure': ['microsoft azure', 'azure ml'],
    'google cloud': ['gcp', 'google cloud platform', 'bigquery', 'vertex ai'],
    # MLOps / DevOps
    'docker': ['containerization', 'containers'],
    'kubernetes': ['k8s', 'container orchestration'],
    'git': ['github', 'gitlab', 'version control', 'bitbucket'],
    'ci/cd': ['continuous integration', 'continuous deployment', 'jenkins', 'github actions'],
    'airflow': ['apache airflow', 'workflow orchestration'],
    'mlflow': ['ml experiment tracking'],
    'fastapi': ['fast api', 'rest api', 'api development'],
    'flask': ['flask api'],
    'django': [],
    # Data Infra
    'spark': ['apache spark', 'pyspark', 'big data'],
    'kafka': ['apache kafka', 'event streaming'],
    'hadoop': ['hdfs', 'mapreduce'],
    'databricks': [],
    'snowflake': [],
    'dbt': ['data build tool'],
    'elasticsearch': ['elastic search'],
    'redis': [],
    'mongodb': ['nosql', 'document database'],
    'postgresql': ['postgres'],
    # Concepts
    'statistics': ['statistical analysis', 'probability', 'hypothesis testing'],
    'linear algebra': ['matrix operations', 'vectors', 'eigenvalues'],
    'optimization': ['gradient descent', 'adam', 'sgd'],
    # FIX: 'a/b testing' kept as-is; 'experimentation' alone is too generic
    'a/b testing': ['split testing', 'ab testing', 'a/b test'],
    'data visualization': ['dashboarding', 'tableau', 'power bi', 'looker', 'power bi'],
    'agile': ['scrum', 'kanban', 'sprint'],
    # FIX: Removed duplicate 'statistics' key (Python silently overwrote it,
    # losing 'hypothesis testing' synonym). Merged into single entry above.
    'communication': ['presentation', 'stakeholder management'],
}

# Flatten synonym → canonical mapping
SYNONYM_MAP = {}
for canonical, synonyms in SKILL_TAXONOMY.items():
    SYNONYM_MAP[canonical] = canonical
    for syn in synonyms:
        SYNONYM_MAP[syn.lower()] = canonical

RESUME_REQUIRED_KEYWORDS = [
    'education', 'experience', 'skills', 'work', 'project',
    'university', 'college', 'degree', 'bachelor', 'master',
    'engineer', 'developer', 'analyst', 'intern', 'certification'
]

print(f'✅ Loaded {len(SKILL_TAXONOMY)} canonical skills, {len(SYNONYM_MAP)} total mappings')

## 📊 Step 4 — Synthetic Dataset Builder

In [ ]:
def build_synthetic_dataset() -> pd.DataFrame:
    """
    Generate a scaled synthetic dataset with ~2500 samples.
    Uses domain-aware pairing and synthetic noise/randomness for diversity:
      - matched cases  (label=1)
      - unrelated cases (label=0)
      - borderline cases (label=0 or 1 by design)
    ML logic and model training are NOT changed — only data volume.
    """

    rng = random.Random(42)  # deterministic noise seed

    # ── Noise pools: slight randomness injected to vary text ──
    _exp_adjectives = ["detail-oriented", "results-driven", "self-motivated", "collaborative",
                       "innovative", "analytical", "pragmatic", "proactive"]
    _edu_variants = ["B.Tech", "M.Tech", "B.Sc.", "M.Sc.", "B.E.", "M.E.", "B.S.", "M.S."]
    _university_variants = ["IIT Bombay", "IIT Delhi", "IIT Madras", "NIT Trichy",
                            "BITS Pilani", "DTU", "VIT", "Manipal University", "SRM University"]
    _cloud_alt = ["AWS", "GCP", "Azure", "AWS/GCP", "GCP/Azure", "AWS and Azure"]
    _cert_phrases = [
        "Certifications: AWS Certified ML Specialty.",
        "Certifications: Google Professional Data Engineer.",
        "Certified: Azure Data Scientist Associate.",
        "Completed: Coursera Deep Learning Specialization.",
        "",  # sometimes no cert mentioned
    ]
    _proj_phrases = [
        "Projects: Fraud detection system ({acc}% AUC), customer segmentation pipeline, A/B test framework.",
        "Projects: Real-time recommendation engine, NLP sentiment classifier ({acc}% F1), data lake on S3.",
        "Projects: Churn prediction model ({acc}% accuracy), ETL pipeline automation, dashboard in Tableau.",
        "Projects: Image classification with ResNet ({acc}% accuracy), MLflow experiment tracking, FastAPI serving.",
        "Projects: Time-series forecasting, feature store design, automated model retraining pipeline.",
    ]

    def _noise(base: str, exp: int) -> str:
        """Inject slight randomness into a resume/JD template string."""
        adj = rng.choice(_exp_adjectives)
        edu = rng.choice(_edu_variants)
        uni = rng.choice(_university_variants)
        cloud = rng.choice(_cloud_alt)
        cert = rng.choice(_cert_phrases)
        proj = rng.choice(_proj_phrases).format(acc=rng.randint(82, 97))
        # substitute placeholders that exist in template, ignore missing ones
        result = base.format(exp=exp)
        result = result.replace("{adj}", adj).replace("{edu}", edu)
        result = result.replace("{uni}", uni).replace("{cloud}", cloud)
        result = result.replace("{cert}", cert).replace("{proj}", proj)
        # Append a small random addendum for text variation
        addendum = rng.choice([
            f" Strong {adj} professional. {cert}",
            f" {edu} from {uni}. {proj}",
            f" Cloud platforms: {cloud}. {cert}",
            f" Known for {adj} approach. {proj}",
            "",
        ])
        return result + addendum

    # ── Resume templates per role (base text, {exp} placeholder) ──
    resume_templates = {
        "data_scientist": [
            "Experienced Data Scientist with {exp} years of experience in machine learning, deep learning, and statistical modeling. "
            "Proficient in Python, TensorFlow, PyTorch, scikit-learn, and SQL. Built predictive models using XGBoost and LightGBM. "
            "Skilled in natural language processing, computer vision, and A/B testing. "
            "Education: M.Sc. in Data Science. Projects: Customer churn prediction, sentiment analysis pipeline using BERT, real-time recommendation engine.",

            "Data Scientist with {exp} years specializing in NLP and generative AI. "
            "Worked with Hugging Face transformers, LLMs, and prompt engineering. "
            "Strong Python, pandas, numpy, matplotlib skills. "
            "Experience deploying models with FastAPI and Docker on AWS SageMaker. "
            "Degree: B.Tech Computer Science. Skills: MLflow, DVC, Airflow, Spark.",

            "Junior Data Scientist with {exp} year of experience. "
            "Familiar with scikit-learn, pandas, matplotlib, and basic SQL. "
            "Completed projects on regression, classification, and clustering. "
            "Education: B.Sc. Statistics. Skills include Python, data visualization, feature engineering.",

            "Senior Data Scientist with {exp} years in e-commerce analytics and recommendation systems. "
            "Proficient in Python, Spark, SQL, XGBoost, and A/B testing. Deep experience with Tableau and Power BI. "
            "Education: M.Sc. Applied Mathematics. Strong statistical modeling and hypothesis testing background.",

            "Data Scientist with {exp} years in healthcare analytics. "
            "Built clinical NLP models using spaCy and BERT. Expertise in survival analysis, logistic regression, and random forests. "
            "Python, R, SQL. Published research on ML in medical imaging. Education: M.D./PhD equivalent.",
        ],
        "ml_engineer": [
            "Machine Learning Engineer with {exp} years building scalable ML systems. "
            "Expert in MLOps: Docker, Kubernetes, Airflow, MLflow, and CI/CD pipelines. "
            "Deployed 10+ production models on AWS and GCP. "
            "Languages: Python, Scala, Go. Frameworks: TensorFlow, PyTorch, FastAPI. "
            "Education: M.Tech AI. Strong in system design and model optimization.",

            "ML Engineer specializing in computer vision and model deployment. "
            "{exp} years experience with OpenCV, YOLO, TensorFlow Lite, and ONNX. "
            "Built edge inference pipelines for IoT devices. "
            "Skills: Python, C++, Docker, Kubernetes, Git, Azure. "
            "Education: B.E. Electronics. Certifications: AWS ML Specialty.",

            "Senior ML Engineer with {exp} years. Expert in distributed training with PyTorch DDP and Spark. "
            "Built feature stores and real-time prediction APIs. "
            "Skills: Python, Kafka, Redis, PostgreSQL, Databricks, Snowflake. "
            "Education: M.Sc. Computer Science.",

            "ML Engineer with {exp} years in NLP and conversational AI. "
            "Built transformer-based pipelines using Hugging Face and deployed on GCP Vertex AI. "
            "Strong Python, FastAPI, Docker skills. Familiar with LangChain and vector databases. "
            "Education: B.Tech Computer Science.",

            "ML Research Engineer with {exp} years. "
            "Focus on reinforcement learning and multi-agent systems. "
            "Published work on policy optimization. PyTorch, JAX, and C++ for performance-critical code. "
            "Education: M.Sc. Computer Science (AI track).",
        ],
        "data_engineer": [
            "Data Engineer with {exp} years of experience designing ETL pipelines and data warehouses. "
            "Expert in Apache Spark, Kafka, Airflow, and dbt. "
            "Built petabyte-scale data lakes on AWS S3 with Redshift and Snowflake. "
            "Languages: Python, SQL, Scala. Skills: Docker, Kubernetes, Git, CI/CD. "
            "Education: B.Tech IT.",

            "Data Engineer with {exp} years focused on real-time streaming architectures. "
            "Used Kafka, Spark Streaming, and Flink for event-driven systems. "
            "Strong in PostgreSQL, MongoDB, Elasticsearch, and Redis. "
            "Python, SQL, and cloud platforms (GCP, Azure). "
            "Degree: M.Sc. Information Systems.",

            "Junior Data Engineer with {exp} year of experience in ETL development. "
            "Skills: Python, SQL, Airflow, dbt, and basic Spark. "
            "Worked with AWS Glue and Redshift for data pipeline development. "
            "Education: B.Sc. Computer Science.",

            "Senior Data Engineer with {exp} years building cloud-native data platforms. "
            "Expert in Databricks, Delta Lake, dbt, and Snowflake. Strong Python, SQL, and Terraform skills. "
            "Led migration of legacy warehouses to modern lakehouse architecture. Education: B.Tech IT.",

            "Data Engineer with {exp} years specializing in analytics engineering. "
            "dbt, Looker, BigQuery, and Airflow expert. Built self-serve analytics platforms. "
            "Python and SQL proficiency. Experience with data quality frameworks and Great Expectations. "
            "Education: M.Sc. Statistics.",
        ],
        "software_engineer": [
            "Software Engineer with {exp} years developing scalable backend systems. "
            "Languages: Java, Python, Go. Frameworks: Spring Boot, Django, FastAPI. "
            "Worked with microservices, REST APIs, and event-driven architectures using Kafka. "
            "Skills: Docker, Kubernetes, CI/CD, PostgreSQL, Redis, Git. "
            "Education: B.Tech Computer Science.",

            "Full-Stack Software Engineer with {exp} years. "
            "Frontend: JavaScript, TypeScript, React, Next.js. Backend: Node.js, Python, Django. "
            "Databases: PostgreSQL, MongoDB, Redis. DevOps: Docker, AWS, GitHub Actions. "
            "Education: B.E. Information Technology.",

            "Backend Engineer with {exp} years specializing in high-performance systems. "
            "C++, Rust, and Python for systems programming. "
            "Experience with distributed systems, gRPC, and message queues. "
            "Education: M.Tech Computer Science.",

            "Software Engineer with {exp} years building fintech platforms. "
            "Java Spring Boot, Kafka, PostgreSQL, and Redis. Strong in TDD and system design. "
            "Experience with PCI-DSS compliance, API gateway design. Education: B.Tech CS.",

            "Mobile & Backend Engineer with {exp} years. "
            "Kotlin for Android, Swift for iOS, Python/FastAPI for backend. "
            "Firebase, MongoDB, and AWS. Experience shipping apps to 1M+ users. "
            "Education: B.Sc. Computer Science.",
        ],
        "devops_engineer": [
            "DevOps Engineer with {exp} years in cloud infrastructure and automation. "
            "Expert in Kubernetes, Docker, Terraform, Ansible, and Jenkins. "
            "Managed AWS, GCP, and Azure infrastructure. "
            "Strong in CI/CD pipelines, monitoring (Prometheus, Grafana), and security. "
            "Education: B.Tech IT.",

            "Site Reliability Engineer with {exp} years. "
            "Skills: Kubernetes, Helm, ArgoCD, Terraform, Python scripting. "
            "Incident management, capacity planning, and chaos engineering. "
            "Education: B.Sc. Computer Science.",

            "Platform Engineer with {exp} years building internal developer platforms. "
            "Expert in Backstage, GitHub Actions, Terraform, and Kubernetes. "
            "Python and Go scripting. FinOps experience reducing cloud costs by 30%. "
            "Education: B.Tech IT.",
        ],
        "unrelated": [
            "Marketing Manager with {exp} years of experience in brand strategy, digital marketing, and campaign management. "
            "Managed social media accounts, SEO optimization, and Google Ads campaigns. "
            "Education: MBA Marketing. Skills: content creation, market research, CRM tools.",

            "HR Manager with {exp} years in talent acquisition, performance management, and employee relations. "
            "Experience with HRMS systems, payroll processing, and labor law compliance. "
            "Education: MBA Human Resources.",

            "Chartered Accountant with {exp} years in auditing, taxation, and financial reporting. "
            "Expertise in GST, income tax, and IFRS. "
            "Education: CA from ICAI, B.Com.",

            "Civil Engineer with {exp} years in structural design and project management. "
            "Worked on highways, bridges, and residential construction. "
            "Education: B.Tech Civil Engineering. Skills: AutoCAD, STAAD Pro, MS Project.",

            "Graphic Designer with {exp} years creating visual assets for digital and print media. "
            "Expert in Adobe Photoshop, Illustrator, InDesign, and Figma. "
            "Education: B.Des Visual Communication.",

            "Operations Manager with {exp} years in supply chain, logistics, and vendor management. "
            "Experience with ERP systems (SAP, Oracle), procurement, and warehouse operations. "
            "Education: MBA Operations. Skills: Six Sigma, lean manufacturing.",

            "Content Writer with {exp} years producing SEO content, blogs, and technical documentation. "
            "Expertise in WordPress, Grammarly, Hemingway Editor, and AP style. "
            "Education: B.A. English Literature.",

            # ── FIX 3 — HARD NEGATIVE RESUMES ────────────────────────────
            # Zero vocabulary overlap with any tech JD — the clearest possible
            # negative examples to teach the model a sharp decision boundary.
            "Digital Marketing Specialist with {exp} years running paid media and influencer campaigns. "
            "Managed Google Ads, Facebook Ads Manager, and HubSpot workflows. "
            "Skills: A/B testing of ad creatives, email automation, and SEO audits. "
            "Education: B.A. Mass Communication. Background in brand communications and media.",  # FIX: Removed negation phrase — TF-IDF treats 'no programming' same as 'programming'

            "Mechanical Design Engineer with {exp} years in product development and CAD modeling. "
            "Designed automotive components using SolidWorks and CATIA. "
            "Hands-on experience with FEA simulation, GD&T, and manufacturing tolerances. "
            "Education: B.Tech Mechanical Engineering. Expertise in physical product design.",  # FIX: Removed negation phrase

            "Licensed Pharmacist with {exp} years of clinical and retail pharmacy experience. "
            "Expert in drug interactions, patient counseling, and dispensing protocols. "
            "Familiar with pharmacy management systems and insurance billing. "
            "Education: B.Pharm, Pharm.D. Expertise in clinical patient care.",  # FIX: Removed negation phrase

            "Primary School Teacher with {exp} years developing K-12 curriculum and classroom management. "
            "Skilled in lesson planning, student assessment, and parent communication. "
            "Education: B.Ed. Expertise in pedagogy and child development.",  # FIX: Removed negation phrase

            "Chef and Culinary Arts professional with {exp} years in fine-dining kitchen management. "
            "Expert in menu design, food cost control, and kitchen staff supervision. "
            "Education: Diploma in Culinary Arts. Expertise in hospitality and food service.",  # FIX: Removed negation phrase
        ],
    }

    # ── Job description templates per role ──
    jd_templates = {
        "data_scientist": [
            "We are looking for a Data Scientist to join our AI team. "
            "Requirements: 2+ years experience in machine learning, deep learning, and statistical modeling. "
            "Proficiency in Python, scikit-learn, TensorFlow or PyTorch. "
            "Experience with NLP, feature engineering, and model evaluation. "
            "Nice to have: SQL, Spark, cloud platforms (AWS/GCP). "
            "Education: M.Sc. or B.Tech in CS/Statistics/Math.",

            "Hiring a Senior Data Scientist specializing in NLP and LLMs. "
            "Must have: 3+ years ML experience, strong Python, Hugging Face, and generative AI skills. "
            "Experience with MLflow, Docker, and API deployment. "
            "Strong statistical background required. Join our NLP research team.",

            "Data Scientist role focused on recommendation systems and personalization. "
            "Requirements: Python, pandas, scikit-learn, A/B testing experience. "
            "SQL proficiency essential. Experience with data visualization (Tableau/Power BI). "
            "Education: B.Tech or M.Sc. in relevant field.",

            "Data Scientist to improve our fraud detection systems. "
            "Required: Python, XGBoost, scikit-learn, SQL, and strong statistics. "
            "Experience with imbalanced datasets and model interpretability (SHAP/LIME). "
            "Nice to have: Spark, AWS SageMaker. 2-5 years experience.",

            "Junior Data Scientist for our analytics team. "
            "Requirements: Python, pandas, matplotlib, basic machine learning. "
            "SQL required. Exposure to scikit-learn and Jupyter notebooks. "
            "Fresh graduates with strong projects welcome.",
        ],
        "ml_engineer": [
            "ML Engineer needed to build and deploy machine learning systems at scale. "
            "Requirements: Python, TensorFlow/PyTorch, Docker, Kubernetes. "
            "Experience with MLOps tools: MLflow, Airflow, DVC. "
            "Cloud experience on AWS or GCP. CI/CD pipeline knowledge essential. "
            "3+ years experience required.",

            "Senior ML Engineer for computer vision products. "
            "Must have: OpenCV, YOLO/object detection, TensorFlow or PyTorch. "
            "Experience with model optimization (ONNX, quantization, pruning). "
            "Cloud deployment on Azure or AWS. C++ is a plus.",

            "ML Platform Engineer to build internal ML infrastructure. "
            "Requirements: Python, Spark, Kafka, Kubernetes, feature stores. "
            "Experience with Databricks, Snowflake, and real-time prediction APIs. "
            "Strong system design and distributed systems knowledge.",

            "ML Engineer for conversational AI and LLM applications. "
            "Must have: Python, Hugging Face, LangChain, vector databases (Pinecone/Weaviate). "
            "FastAPI and Docker for model serving. GCP or AWS deployment. "
            "2+ years ML engineering experience.",

            "Production ML Engineer to maintain and improve recommendation systems. "
            "Requirements: Python, scikit-learn, Spark MLlib, Kafka, and Redis. "
            "Experience with A/B testing and online learning. Strong SQL. "
            "3+ years in ML production environments.",
        ],
        "data_engineer": [
            "Data Engineer to design and maintain our data infrastructure. "
            "Requirements: Python, SQL, Apache Spark, Airflow, and dbt. "
            "Experience building data pipelines and warehouses on AWS (S3, Redshift) or Snowflake. "
            "3+ years in data engineering. Docker and Kubernetes knowledge preferred.",

            "Senior Data Engineer for real-time streaming platform. "
            "Must have: Kafka, Spark Streaming, PostgreSQL, and Python. "
            "Experience with event-driven architectures and cloud data platforms (GCP/Azure). "
            "Strong SQL skills and understanding of data modeling.",

            "Data Engineer to work on our analytics platform. "
            "Skills required: SQL, Python, ETL development, and dbt. "
            "Familiarity with Airflow and cloud data warehouses. "
            "Junior-to-mid level, 1-3 years experience.",

            "Analytics Engineer to build and maintain our dbt models. "
            "Requirements: dbt, SQL, BigQuery or Snowflake, Looker, and Python. "
            "Experience with data quality, testing, and documentation practices. "
            "Bonus: Airflow, Great Expectations. 2+ years experience.",

            "Data Engineer for our lakehouse modernization project. "
            "Must have: Databricks, Delta Lake, Spark, Python, and SQL. "
            "Terraform for infrastructure. Experience migrating legacy Hadoop workloads. "
            "3-5 years data engineering experience.",
        ],
        "software_engineer": [
            "Backend Software Engineer for our core platform team. "
            "Requirements: Java or Go, microservices, REST APIs, Kafka. "
            "PostgreSQL, Redis, Docker, Kubernetes. CI/CD experience. "
            "3+ years backend development. System design skills required.",

            "Full-Stack Engineer to build our web platform. "
            "Frontend: React, TypeScript. Backend: Node.js or Python/Django. "
            "PostgreSQL, MongoDB, AWS deployment. "
            "2+ years full-stack experience. Git workflow required.",

            "Software Engineer specializing in high-performance systems. "
            "Languages: C++ or Rust, Python scripting. "
            "Distributed systems experience, gRPC, and message queues. "
            "Education: M.Tech or B.Tech CS.",

            "Backend Engineer for fintech payment platform. "
            "Requirements: Java Spring Boot, Kafka, PostgreSQL, Redis. "
            "Strong in security, PCI compliance, and high-throughput system design. "
            "3+ years backend experience. Bonus: Python.",

            "Software Engineer — API Platform team. "
            "Requirements: Python or Go, REST and GraphQL APIs, PostgreSQL. "
            "Docker, Kubernetes, and CI/CD. Good understanding of OAuth and API security. "
            "2-4 years experience. Remote-friendly.",
        ],
        "devops_engineer": [
            "DevOps Engineer for our cloud platform. "
            "Requirements: Kubernetes, Docker, Terraform, Jenkins/GitHub Actions. "
            "AWS or GCP infrastructure management. Monitoring with Prometheus/Grafana. "
            "Strong scripting skills (Python/Bash). 3+ years experience.",

            "SRE to maintain reliability and performance of our infrastructure. "
            "Must have: Kubernetes, Helm, Terraform, Python. "
            "Incident management, SLA/SLO management, and chaos engineering experience. "
            "Strong cloud background (AWS/GCP/Azure).",

            "Platform Engineer to build and improve our internal developer platform. "
            "Requirements: Kubernetes, Terraform, GitHub Actions, ArgoCD, Go or Python. "
            "Experience with Backstage, FinOps, and cost optimization. "
            "Bonus: service mesh (Istio/Linkerd). 4+ years DevOps experience.",
        ],
        "unrelated": [
            "Marketing Specialist to develop and execute digital marketing campaigns. "
            "Requirements: SEO, Google Ads, social media management, content creation. "
            "Experience with CRM tools and market research. MBA preferred.",

            "HR Business Partner for talent acquisition and organizational development. "
            "Requirements: recruitment, performance management, HRMS, labor law knowledge.",

            "Financial Analyst for our accounting and finance team. "
            "Requirements: financial modeling, Excel, accounting principles, GST, tax compliance.",

            "Civil Project Manager for infrastructure projects. "
            "Requirements: AutoCAD, project scheduling, structural design, site supervision.",

            "Content Marketing Manager. Requirements: blog writing, SEO, email campaigns, "
            "HubSpot, Google Analytics. Strong English writing and editorial skills.",

            # ── FIX 3 — HARD NEGATIVE JDs ────────────────────────────────
            "Brand Strategy Director for consumer goods company. "
            "Must have: market segmentation, brand architecture, consumer insight research, "
            "campaign planning, and agency management. MBA in Marketing required. "
            "Brand management and consumer research experience essential.",  # FIX: Removed negation phrase

            "Structural Engineer for infrastructure consultancy. "
            "Requirements: RCC design, AutoCAD, STAAD Pro, IS code knowledge, "
            "site inspection, and project scheduling. B.Tech Civil Engineering required. "
            "Experience in structural planning and site management required.",  # FIX: Removed negation phrase

            "Pediatric Nurse Practitioner for hospital ward. "
            "Requirements: clinical assessment, medication administration, patient care, "
            "EMR documentation, and family counseling. BSN or MSN required. "
            "Clinical certification and patient care experience required.",  # FIX: Removed negation phrase
        ],
    }

    samples = []
    tech_roles = ["data_scientist", "ml_engineer", "data_engineer", "software_engineer", "devops_engineer"]

    # ── 1. STRONG MATCHES (label=1): Same role pairs, multiple exp levels + noise repeats ──
    for role in tech_roles:
        r_templates = resume_templates[role]
        j_templates = jd_templates[role]
        for r_tmpl in r_templates:
            for j_tmpl in j_templates:
                for exp in [1, 2, 3, 4, 5, 6]:
                    # Base sample
                    r = r_tmpl.format(exp=exp)
                    samples.append({"resume": r, "job_description": j_tmpl, "label": 1})
                    # Noise variant (slightly randomized text)
                    r_noisy = _noise(r_tmpl, exp)
                    if r_noisy != r:
                        samples.append({"resume": r_noisy, "job_description": j_tmpl, "label": 1})

    # ── 2. UNRELATED PAIRS (label=0): Unrelated resume + tech JD, with noise ──
    unrelated_resumes = resume_templates["unrelated"]
    unrelated_jds     = jd_templates["unrelated"]

    for r_tmpl in unrelated_resumes:
        for exp in [1, 2, 3, 5, 7]:
            r = r_tmpl.format(exp=exp)
            # vs all tech JDs
            for role in tech_roles:
                for j_tmpl in jd_templates[role]:
                    samples.append({"resume": r, "job_description": j_tmpl, "label": 0})
            # vs unrelated JDs (non-tech resume, non-tech JD → 0)
            for j_tmpl in unrelated_jds:
                samples.append({"resume": r, "job_description": j_tmpl, "label": 0})
            # noise repeat
            r_noisy = _noise(r_tmpl, exp)
            if r_noisy != r:
                for role in ["data_scientist", "ml_engineer"]:
                    samples.append({"resume": r_noisy, "job_description": jd_templates[role][0], "label": 0})

    # ── 3. BORDERLINE CASES ──
    # Cross-role tech mismatch (distant roles → label=0)
    for i, role_a in enumerate(tech_roles):
        for j, role_b in enumerate(tech_roles):
            if abs(i - j) >= 2:
                for exp in [1, 2, 3, 4]:
                    r = resume_templates[role_a][0].format(exp=exp)
                    for j_tmpl in jd_templates[role_b][:2]:
                        samples.append({"resume": r, "job_description": j_tmpl, "label": 0})

    # Junior resume vs senior JD → label=0 (borderline)
    junior_ds  = resume_templates["data_scientist"][2].format(exp=1)
    senior_jds = [jd_templates["data_scientist"][0], jd_templates["ml_engineer"][0]]
    for j_tmpl in senior_jds:
        for _ in range(8):
            samples.append({"resume": junior_ds, "job_description": j_tmpl, "label": 0})

    # Adjacent-role cross matches → label=1 (partial fit)
    adjacent_pairs = [
        ("data_scientist", "ml_engineer"),
        ("ml_engineer", "data_scientist"),
        ("data_engineer", "software_engineer"),
        ("software_engineer", "devops_engineer"),
    ]
    for role_a, role_b in adjacent_pairs:
        r = resume_templates[role_a][0].format(exp=3)
        for j_tmpl in jd_templates[role_b][:2]:
            for _ in range(6):
                samples.append({"resume": r, "job_description": j_tmpl, "label": 1})

    # ── 4. REAL-DATA AUGMENTATION PLACEHOLDERS (100–200 extra clean matched rows) ──
    # Add more same-role matched rows with varied experience to reach target size
    for role in tech_roles:
        r_templates = resume_templates[role]
        j_templates = jd_templates[role]
        for r_tmpl in r_templates:
            for j_tmpl in j_templates:
                for exp in range(1, 11):   # 1–10 years
                    r_noisy = _noise(r_tmpl, exp)
                    samples.append({"resume": r_noisy, "job_description": j_tmpl, "label": 1})

    df = pd.DataFrame(samples)
    df = df.drop_duplicates(subset=["resume", "job_description"]).reset_index(drop=True)
    print(f"[INFO] Built synthetic dataset: {len(df)} samples "
          f"| label=1: {(df['label']==1).sum()} | label=0: {(df['label']==0).sum()}")
    return df


def augment_with_real_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Attempt to load and augment with real HuggingFace dataset.
    Falls back gracefully if unavailable.
    """
    if not HF_AVAILABLE:
        print("[INFO] datasets library not available. Skipping real data augmentation.")
        return df

# Build dataset
print('Building dataset...')
df = build_synthetic_dataset()
print(f'Label distribution: {df["label"].value_counts().to_dict()}')
augmented = augment_with_real_data(df)
if augmented is not None:
    df = augmented
print(f'\nFinal dataset: {len(df)} samples')
df.head(3)


## 🔧 Step 5 — Text Preprocessor & Resume Validator

In [ ]:
class TextPreprocessor:
    """Handles all text normalization:
    lowercase → punctuation removal → stopword removal → lemmatization
    """
    def __init__(self):
        self.lemmatizer = WordNetLemmatizer()
        self.stop_words = set(stopwords.words('english'))
        self.stop_words -= {'no', 'not', 'c', 'r'}  # Keep important tech tokens

    def clean(self, text: str) -> str:
        if not text or not isinstance(text, str):
            return ''
        text = text.lower()
        # FIX: Preserve version numbers like python3, java8, word2vec before digit removal
        text = re.sub(r'([a-z]+)(\d+)', r'\1', text)  # strip trailing digits: python3→python
        # Split hyphenated terms (scikit-learn → scikit learn)
        text = re.sub(r'(?<=[a-z])-(?=[a-z])', ' ', text)
        # Keep letters, spaces, / # + (for C++, C#, CI/CD, A/B)
        text = re.sub(r'[^a-z\s/#+]', ' ', text)
        tokens = text.split()
        # FIX: len > 2 instead of > 1 to avoid single-char noise ('s', 'a', 'r' residuals)
        # but explicitly keep known 2-char tech tokens
        KEEP_SHORT = {'go', 'ai', 'ml', 'dl', 'r', 'c'}
        tokens = [
            self.lemmatizer.lemmatize(t)
            for t in tokens
            if (t not in self.stop_words) and (len(t) > 2 or t in KEEP_SHORT)
        ]
        return ' '.join(tokens)


class ResumeValidator:
    """Validates that uploaded document is a genuine resume."""
    def validate(self, text: str) -> tuple:
        if not text or len(text.strip()) < 100:
            return False, '❌ Document is too short or empty. Please upload a valid resume.'
        text_lower = text.lower()
        # FIX: Expand keyword list to cover B.Tech, M.Tech, B.E., M.E. degree formats
        extended_keywords = RESUME_REQUIRED_KEYWORDS + [
            'b.tech', 'm.tech', 'b.e.', 'm.e.', 'b.sc', 'm.sc', 'b.s.', 'm.s.',
            'phd', 'mba', 'diploma', 'graduated', 'professional', 'summary'
        ]
        found = [kw for kw in extended_keywords if kw in text_lower]
        # FIX: Require at least 3 matches for stricter validation (was 2)
        if len(found) < 3:
            return False, (
                f'❌ This does not appear to be a resume. '
                f'Missing key sections. Found only: {found or ["none"]}. '
                f'Please upload a document containing education, experience, and skills sections.'
            )
        return True, f'✅ Valid resume detected. Found: {", ".join(found[:5])}.'


class FileParser:
    """Extracts raw text from PDF and DOCX files."""
    def extract(self, file_path: str) -> str:
        ext = os.path.splitext(file_path)[-1].lower()
        if ext == '.pdf':
            return self._parse_pdf(file_path)
        elif ext in ['.docx', '.doc']:
            return self._parse_docx(file_path)
        else:
            raise ValueError(f'Unsupported format: {ext}. Please upload PDF or DOCX.')

    def _parse_pdf(self, path: str) -> str:
        text = []
        with open(path, 'rb') as f:
            reader = PyPDF2.PdfReader(f)
            for page in reader.pages:
                t = page.extract_text()
                if t:
                    text.append(t)
        return '\n'.join(text)

    def _parse_docx(self, path: str) -> str:
        doc = docx.Document(path)
        return '\n'.join([p.text for p in doc.paragraphs if p.text.strip()])


class SkillExtractor:
    """
    Extracts and maps skills using:
    1. Canonical skill keyword matching
    2. Synonym mapping (80+ tech aliases)
    """
    def __init__(self):
        self.canonical_skills = list(SKILL_TAXONOMY.keys())
        self.synonym_map = SYNONYM_MAP

    def _normalize_to_skills(self, text: str) -> set:
        text_lower = text.lower()
        found = set()
        for skill in self.canonical_skills:
            if skill in text_lower:
                found.add(skill)
        for synonym, canonical in self.synonym_map.items():
            if synonym in text_lower:
                found.add(canonical)
        return found

    def extract(self, resume_text: str, jd_text: str) -> dict:
        resume_skills = self._normalize_to_skills(resume_text)
        jd_skills = self._normalize_to_skills(jd_text)
        return {
            'resume_skills': sorted(resume_skills),
            'jd_skills': sorted(jd_skills),
            'matched': sorted(resume_skills & jd_skills),
            'missing': sorted(jd_skills - resume_skills),
            'extra': sorted(resume_skills - jd_skills),
        }


# Quick test
preprocessor = TextPreprocessor()
test = 'Experienced Machine Learning Engineer with Python, TensorFlow, and AWS skills.'
print(f'Raw:       {test}')
print(f'Processed: {preprocessor.clean(test)}')

## 🤖 Step 6 — Model Training with Cross-Validation

In [ ]:
class ModelTrainer:
    """
    Trains 4 ML models using TF-IDF + StratifiedKFold (cv=5).
    Selects best model by mean F1-score.
    """
    def __init__(self):
        self.preprocessor = TextPreprocessor()
        self.vectorizer = TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 2),
            sublinear_tf=True,
            min_df=2,        # FIX: was 1 — keeping singletons causes overfitting to noise tokens
            max_df=0.95,     # FIX: ignore terms in >95% of docs (too common to be discriminative)
        )
        self.models = {}
        self.cv_results = {}
        self.best_model_name = None
        self.best_model = None
        self.best_f1 = 0.0
        self.X_train_tfidf = None
        self.y_train = None

    def _get_model_definitions(self) -> dict:
        return {
            'Logistic Regression': Pipeline([
                ('clf', LogisticRegression(max_iter=1000, C=1.0, random_state=42))
            ]),
            'Naive Bayes': Pipeline([
                ('clf', MultinomialNB(alpha=0.5))
            ]),
            'SVM': Pipeline([
                ('clf', CalibratedClassifierCV(LinearSVC(C=1.0, max_iter=2000, random_state=42)))
            ]),
            'Random Forest': Pipeline([
                ('clf', RandomForestClassifier(n_estimators=200, max_depth=15,
                                              random_state=42, n_jobs=-1))
            ]),
        }

    def _make_structured_text(self, resume: str, jd: str) -> str:
        """
        FIX 1 — FEATURE SEPARATION
        Prefix each section with a sentinel token so TF-IDF builds
        distinct n-gram features per region (resumesection/jdsection).
        The same format MUST be used at inference time.
        """
        r_clean  = self.preprocessor.clean(resume)
        jd_clean = self.preprocessor.clean(jd)
        # Repeat twice so sentinels survive min_df filtering
        return f"resumesection resumesection {r_clean} jdsection jdsection {jd_clean}"

    def train(self, df: pd.DataFrame):
        print(f'Dataset: {len(df)} samples | Distribution: {df["label"].value_counts().to_dict()}')

        # FIX 2 — DATASET BALANCING
        # Undersample majority class to 1:1 ratio so the model cannot
        # "win" by always predicting the dominant label.
        n_pos = (df["label"] == 1).sum()
        n_neg = (df["label"] == 0).sum()
        minority_n = min(n_pos, n_neg)
        df_pos = df[df["label"] == 1].sample(n=minority_n, random_state=42)
        df_neg = df[df["label"] == 0].sample(n=minority_n, random_state=42)
        df_balanced = (pd.concat([df_pos, df_neg])
                         .sample(frac=1, random_state=42)
                         .reset_index(drop=True))
        print(f'[INFO] Balanced: {len(df_balanced)} samples (1: {minority_n}, 0: {minority_n})')

        # FIX 1 — apply structured separator to every row
        df_balanced['cleaned'] = df_balanced.apply(
            lambda row: self._make_structured_text(row['resume'], row['job_description']),
            axis=1
        )

        X_text = df_balanced['cleaned'].values
        y      = df_balanced['label'].values

        X_tfidf = self.vectorizer.fit_transform(X_text)
        self.X_train_tfidf = X_tfidf
        self.y_train = y

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        model_defs = self._get_model_definitions()

        # FIX: Collect held-out predictions for confusion matrix (not training data)
        self.cv_holdout_preds = None   # will store best model's OOF predictions
        self.cv_holdout_true  = None

        print('\nRunning 5-fold StratifiedKFold cross-validation...')
        print('-' * 60)
        
        # First pass: evaluate all models via CV to find the winner
        cv_scores = {}
        for name, pipeline in model_defs.items():
            cv_out = cross_validate(
                pipeline, X_tfidf, y,
                cv=skf, scoring='f1',
                return_train_score=True
            )
            mean_f1  = float(np.mean(cv_out['test_score']))
            std_f1   = float(np.std(cv_out['test_score']))
            train_f1 = float(np.mean(cv_out['train_score']))
            cv_scores[name] = mean_f1
            self.cv_results[name] = {
                'mean_f1': mean_f1, 'std_f1': std_f1,
                'train_f1': train_f1,
                'fold_scores': cv_out['test_score'].tolist()
            }
            print(f'  {name:25s} | F1: {mean_f1:.4f} ± {std_f1:.4f}')

        # FIX: Determine best model AFTER all CV scores collected, then print ★ correctly
        self.best_model_name = max(cv_scores, key=cv_scores.get)
        self.best_f1 = cv_scores[self.best_model_name]
        print('-' * 60)
        print(f'\n✅ Best Model: {self.best_model_name} (F1={self.best_f1:.4f})')

        # Second pass: refit all models on full data; collect OOF preds for best model
        from sklearn.model_selection import cross_val_predict
        for name, pipeline in model_defs.items():
            # FIX: Collect out-of-fold predictions for best model → honest confusion matrix
            if name == self.best_model_name:
                oof_preds = cross_val_predict(pipeline, X_tfidf, y, cv=skf)
                self.cv_holdout_preds = oof_preds
                self.cv_holdout_true  = y
            pipeline.fit(X_tfidf, y)
            self.models[name] = pipeline
            if name == self.best_model_name:
                self.best_model = pipeline

    def get_cv_summary(self) -> pd.DataFrame:
        rows = []
        for name, stats in self.cv_results.items():
            rows.append({
                'Model': name,
                'Mean F1': round(stats['mean_f1'], 4),
                'Std F1': round(stats['std_f1'], 4),
                'Train F1': round(stats['train_f1'], 4),
                'Best Fold F1': round(max(stats['fold_scores']), 4),
            })
        return pd.DataFrame(rows).sort_values('Mean F1', ascending=False).reset_index(drop=True)


# Train all models
trainer = ModelTrainer()
trainer.train(df)

print('\nModel Comparison Table:')
trainer.get_cv_summary()

## 🎯 Step 7 — Resume Matcher & Scoring Engine

In [ ]:
class ResumeMatcher:
    """
    Combined scoring:
    Final Score = 0.6 × ML Probability + 0.4 × Cosine Similarity
    """
    def __init__(self, trainer: ModelTrainer):
        self.trainer = trainer
        self.preprocessor = TextPreprocessor()
        self.skill_extractor = SkillExtractor()
        self.validator = ResumeValidator()

    def match(self, resume_text: str, jd_text: str) -> dict:
        """
        FIX 4 — SCORE CORRECTION
        At inference we now use the SAME structured-separator format
        (_make_structured_text) that was used during training.
        Previously raw concatenation was fed to a vectorizer trained on
        sentinel-prefixed text — a distribution mismatch that inflated
        model_proba for unrelated inputs.

        Additional low-cosine cap: if cosine < 5 %, vocabularies share
        almost nothing, so we cap model_score at 35 % regardless of what
        the ML model believes.  This prevents a confident-but-wrong model
        from pushing unrelated pairs over the Good-Fit threshold.

        FIX 5 — THRESHOLD LOGIC
        Good Fit requires score >= 55 % AND cosine >= 5 %.

        FIX 6 — DEBUG VISIBILITY
        A single log line prints all three components to the terminal.
        """
        resume_clean = self.preprocessor.clean(resume_text)
        jd_clean     = self.preprocessor.clean(jd_text)

        # FIX 4a — build the same structured text used during training
        structured_text = (
            f"resumesection resumesection {resume_clean} "
            f"jdsection jdsection {jd_clean}"
        )

        # Cosine similarity — on raw cleaned sections (no sentinels)
        # so the angle purely reflects vocabulary overlap
        try:
            tfidf_matrix = self.trainer.vectorizer.transform([resume_clean, jd_clean])
            cosine_sim = float(cosine_similarity(tfidf_matrix[0], tfidf_matrix[1])[0][0])
        except Exception as e:  # FIX: bare except → except Exception
            cosine_sim = 0.0
            print(f'[WARN] Cosine similarity failed: {e}')

        # ML model probability — on structured text (matches training)
        try:
            structured_vec = self.trainer.vectorizer.transform([structured_text])
            proba = self.trainer.best_model.predict_proba(structured_vec)[0]
            model_score = float(proba[1])
        except Exception as e:  # FIX: bare except → except Exception
            model_score = 0.0
            print(f'[WARN] Model prediction failed: {e}')

        # FIX 4b — low-cosine penalty: cap model_score when overlap ≈ 0
        LOW_COSINE_THRESHOLD = 0.05   # below this → near-zero vocab overlap
        LOW_COSINE_PROBA_CAP = 0.35   # cap model_score here in that case
        if cosine_sim < LOW_COSINE_THRESHOLD:
            model_score = min(model_score, LOW_COSINE_PROBA_CAP)

        # Weighted combination (0.6 / 0.4 unchanged from original design)
        final_score = 0.6 * model_score + 0.4 * cosine_sim
        final_pct = round(final_score * 100, 2)

        # FIX 5 — dual condition threshold
        SCORE_THRESHOLD = 55.0   # raised from 50 for tighter boundary
        COSINE_MIN_PCT  = 5.0    # raw vocabulary overlap must be >= 5 %
        is_good_fit = (final_pct >= SCORE_THRESHOLD) and (cosine_sim * 100 >= COSINE_MIN_PCT)

        # FIX 6 — debug log (visible in Colab terminal and Gradio logs)
        print(
            f"[MATCH DEBUG] "
            f"cosine={cosine_sim*100:.1f}%  "
            f"ml_proba={model_score*100:.1f}%  "
            f"final={final_pct:.1f}%  "
            f"verdict={'GOOD FIT' if is_good_fit else 'NOT FIT'}"
        )

        skill_info  = self.skill_extractor.extract(resume_text, jd_text)
        suggestions = self._generate_suggestions(final_pct, skill_info['missing'], skill_info['matched'])

        return {
            'model_score': round(model_score * 100, 2),
            'cosine_sim':  round(cosine_sim * 100, 2),
            'final_score': final_pct,
            'is_good_fit': is_good_fit,
            'skill_info':  skill_info,
            'suggestions': suggestions,
            'best_model':  self.trainer.best_model_name,
            'best_f1':     round(self.trainer.best_f1, 4),
            'cv_results':  self.trainer.cv_results,
        }

    def _generate_suggestions(self, score: float, missing: list, matched: list) -> list:
        suggestions = []
        if missing:
            top_missing = ', '.join(missing[:5])  # FIX: was chr(44).join — unreadable hack
            suggestions.append(f'🎯 Add these missing skills: **{top_missing}**')
        if score < 40:
            suggestions.append(
                '📝 Your resume needs significant alignment. '
                'Tailor your summary section to mirror the job description language.'
            )
        elif score < 60:
            suggestions.append(
                '✏️ Moderate match. Highlight projects where you used the required technologies more explicitly.'
            )
        else:
            suggestions.append(
                '✅ Strong match! Quantify achievements with metrics (e.g., "Improved accuracy by 15%").'
            )
        if len(matched) < 3:
            suggestions.append(
                '🔍 Use the exact technology names from the JD (e.g., "TensorFlow" not just "deep learning frameworks").'
            )
        suggestions.append('📄 Add a 2-3 line professional summary targeting this specific role.')
        if missing:
            cert_skills = ', '.join(missing[:3])  # FIX: moved join outside f-string
            suggestions.append(f'📚 Consider certifications for: {cert_skills} (Coursera, Udemy, or official docs).')
        return suggestions[:4]


matcher = ResumeMatcher(trainer)
print('✅ ResumeMatcher initialized!')

## 📈 Step 8 — Visualization Engine

In [ ]:
class VisualizationEngine:
    """Generates F1 bar chart, confusion matrix, and score gauge."""

    def __init__(self, trainer: ModelTrainer):
        self.trainer = trainer

    def f1_bar_chart(self) -> plt.Figure:
        summary = self.trainer.get_cv_summary()
        fig, ax = plt.subplots(figsize=(9, 5))
        fig.patch.set_facecolor('#0f1117')
        ax.set_facecolor('#1a1d27')

        colors = ['#00d4ff', '#7c3aed', '#10b981', '#f59e0b']
        bars = ax.bar(summary['Model'], summary['Mean F1'],
                      color=colors[:len(summary)], width=0.5,
                      edgecolor='#ffffff20', linewidth=0.8)
        ax.errorbar(range(len(summary)), summary['Mean F1'],
                    yerr=summary['Std F1'], fmt='none',
                    color='white', capsize=5, linewidth=1.5)
        for bar, val in zip(bars, summary['Mean F1']):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.015,
                    f'{val:.4f}', ha='center', va='bottom',
                    color='white', fontsize=10, fontweight='bold')

        best_idx = summary['Model'].tolist().index(self.trainer.best_model_name)
        bars[best_idx].set_edgecolor('#ffffff')
        bars[best_idx].set_linewidth(2.5)

        ax.set_ylim(0, 1.12)
        ax.set_ylabel('Mean F1 Score (CV=5)', color='white', fontsize=11)
        ax.set_title('📊 Model Comparison — Cross-Validation F1 Scores',
                     color='white', fontsize=13, fontweight='bold', pad=15)
        ax.tick_params(colors='white')
        for spine in ['top', 'right']:
            ax.spines[spine].set_visible(False)
        for spine in ['left', 'bottom']:
            ax.spines[spine].set_color('#ffffff30')
        ax.set_xticklabels(summary['Model'], color='white', fontsize=10)

        best_patch = mpatches.Patch(facecolor='white', edgecolor='white',
                                    label=f'★ Best: {self.trainer.best_model_name}')
        ax.legend(handles=[best_patch], facecolor='#1a1d27',
                  edgecolor='#ffffff30', labelcolor='white', fontsize=9)
        plt.tight_layout()
        return fig

    def confusion_matrix_chart(self) -> plt.Figure:
        # FIX: Was predicting on TRAINING data → always shows near-perfect results.
        # Now uses out-of-fold (OOF) predictions = honest held-out performance.
        if (self.trainer.cv_holdout_preds is not None and
                self.trainer.cv_holdout_true is not None):
            y_pred = self.trainer.cv_holdout_preds
            y_true = self.trainer.cv_holdout_true
        else:
            # Fallback if OOF not available
            y_pred = self.trainer.best_model.predict(self.trainer.X_train_tfidf)
            y_true = self.trainer.y_train
        cm = confusion_matrix(y_true, y_pred)

        fig, ax = plt.subplots(figsize=(6, 5))
        fig.patch.set_facecolor('#0f1117')
        ax.set_facecolor('#1a1d27')

        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                    linewidths=1, linecolor='#ffffff20',
                    xticklabels=['Not Fit (0)', 'Good Fit (1)'],
                    yticklabels=['Not Fit (0)', 'Good Fit (1)'],
                    annot_kws={'size': 14, 'weight': 'bold', 'color': 'white'},
                    cbar_kws={'shrink': 0.8})
        ax.set_title(f'🧩 Confusion Matrix — {self.trainer.best_model_name}',
                     color='white', fontsize=12, fontweight='bold', pad=12)
        ax.set_xlabel('Predicted Label', color='white', fontsize=10)
        ax.set_ylabel('True Label', color='white', fontsize=10)
        ax.tick_params(colors='white')
        ax.yaxis.set_tick_params(rotation=0)
        plt.tight_layout()
        return fig

    def score_gauge_chart(self, score: float) -> plt.Figure:
        fig, ax = plt.subplots(figsize=(6, 3.5), subplot_kw={'polar': False})
        fig.patch.set_facecolor('#0f1117')
        ax.set_facecolor('#0f1117')
        ax.axis('off')

        theta = np.linspace(np.pi, 0, 300)
        ax.plot(np.cos(theta), np.sin(theta), color='#ffffff15', linewidth=30)

        fill_theta = np.linspace(np.pi, np.pi - (score/100)*np.pi, 300)
        # FIX: Was score>=60 for green, but Good Fit threshold is 55%. Aligned.
        color = '#10b981' if score >= 55 else ('#f59e0b' if score >= 35 else '#ef4444')
        ax.plot(np.cos(fill_theta), np.sin(fill_theta), color=color, linewidth=30)

        ax.text(0, 0.05, f'{score:.1f}%', ha='center', va='center',
                fontsize=28, fontweight='bold', color='white')
        ax.text(0, -0.25, 'Match Score', ha='center', va='center',
                fontsize=11, color='#aaaaaa')
        ax.text(-1.15, -0.15, '0%', color='#aaaaaa', fontsize=9)
        ax.text(1.05, -0.15, '100%', color='#aaaaaa', fontsize=9)
        ax.text(0, 1.1, '50%', ha='center', color='#aaaaaa', fontsize=9)
        ax.set_xlim(-1.4, 1.4)
        ax.set_ylim(-0.5, 1.3)
        plt.tight_layout()
        return fig


viz = VisualizationEngine(trainer)

# Preview charts
print('Generating model comparison chart...')
fig = viz.f1_bar_chart()
plt.show()

print('Generating confusion matrix...')
fig2 = viz.confusion_matrix_chart()
plt.show()
print('✅ Charts ready!')

## 🚀 Step 9 — Launch Gradio UI

In [ ]:
file_parser = FileParser()
validator = ResumeValidator()


def process(resume_file, jd_text: str):
    """Main processing handler called by Gradio."""
    if resume_file is None:
        return ('⚠️ Please upload a resume file (PDF or DOCX).',
                '', '', '', '', '', None, None, None)
    if not jd_text or len(jd_text.strip()) < 100:  # FIX: was 30 chars — too permissive; JDs need substance
        return ('⚠️ Please provide a job description (at least 30 characters).',
                '', '', '', '', '', None, None, None)

    try:
        resume_text = file_parser.extract(resume_file.name)
    except Exception as e:
        return (str(e), '', '', '', '', '', None, None, None)

    is_valid, val_msg = validator.validate(resume_text)
    if not is_valid:
        return (val_msg, '', '', '', '', '', None, None, None)

    result = matcher.match(resume_text, jd_text)
    score = result['final_score']
    fit_label = '🟢 GOOD FIT' if result['is_good_fit'] else '🔴 NOT A FIT'

    score_md = f"""
## 🎯 Match Score: **{score:.1f}%**

| Metric | Value |
|--------|-------|
| 🤖 ML Model Probability | {result['model_score']:.1f}% |
| 📐 Cosine Similarity | {result['cosine_sim']:.1f}% |
| 🔀 Combined Final Score | **{score:.1f}%** |
| 📌 Verdict | **{fit_label}** |
"""

    cv = result['cv_results'][result['best_model']]
    model_md = f"""
## 🧠 Model Intelligence

**Best Model:** `{result['best_model']}`

| Metric | Value |
|--------|-------|
| Mean F1 Score (CV=5) | `{result['best_f1']:.4f}` |
| Std Deviation | `±{cv['std_f1']:.4f}` |
| Train F1 Score | `{cv['train_f1']:.4f}` |
| Best Fold F1 | `{max(cv['fold_scores']):.4f}` |
| Fold Scores | `{[round(x,3) for x in cv['fold_scores']]}` |

*Formula: 0.6 × ML Probability + 0.4 × Cosine Similarity  |  Good Fit: score ≥ 55% AND cosine ≥ 5%*
"""

    si = result['skill_info']
    matched_str = ', '.join([f'`{s}`' for s in si['matched']]) or '*None detected*'
    missing_str = ', '.join([f'`{s}`' for s in si['missing']]) or '*None — you have all required skills!*'
    extra_str = ', '.join([f'`{s}`' for s in si['extra'][:5]]) or '*None*'

    skills_md = f"""
## 🧩 Skill Analysis

### ✅ Matched Skills ({len(si['matched'])})
{matched_str}

### ❌ Missing Skills ({len(si['missing'])})
{missing_str}

### ➕ Extra Skills ({len(si['extra'])})
{extra_str}
"""

    sugg_md = '## 💡 Improvement Suggestions\n\n'
    for i, s in enumerate(result['suggestions'], 1):
        sugg_md += f'{i}. {s}\n\n'

    cv_df = trainer.get_cv_summary()
    cv_md = f'## 📊 Model Comparison Table\n\n{cv_df.to_markdown(index=False)}'

    gauge_fig = viz.score_gauge_chart(score)
    f1_fig = viz.f1_bar_chart()
    cm_fig = viz.confusion_matrix_chart()

    return (val_msg, score_md, model_md, skills_md, sugg_md, cv_md,
            gauge_fig, f1_fig, cm_fig)


# ─── Build Gradio UI ───
custom_css = """
/* ── Force dark base ── */
body, .gradio-container, .main, .wrap {
    background: #0f1117 !important;
    color: #e0e0e0 !important;
    font-family: 'DM Sans', 'Segoe UI', sans-serif !important;
}

/* ── Tabs ── */
.tab-nav { background: #1a1d27 !important; border-radius: 12px !important; }
.tab-nav button { color: #aaaaaa !important; font-weight: 600; font-size: 14px; background: transparent !important; }
.tab-nav button.selected { color: #00d4ff !important; border-bottom: 2px solid #00d4ff !important; background: #0f1117 !important; }

/* ── Panels & blocks ── */
.block, .panel, .gr-box, .gr-form, .form, .gap, .contain {
    background: #13161f !important;
    border-color: #ffffff12 !important;
}

/* ── Markdown / prose text ── */
.prose, .prose p, .prose li, .md, .md p, .md li,
.markdown-body, .markdown-body p, .markdown-body li {
    color: #e0e0e0 !important;
}
.prose h1, .prose h2, .md h1, .md h2, .markdown-body h1, .markdown-body h2 {
    color: #00d4ff !important;
    border-bottom: 1px solid #ffffff18 !important;
    padding-bottom: 4px !important;
}
.prose h3, .md h3, .markdown-body h3 {
    color: #a78bfa !important;
}

/* ── Tables ── */
.prose table, .md table, .markdown-body table,
table { background: #1a1d27 !important; border-collapse: collapse !important; width: 100% !important; }
.prose th, .md th, .markdown-body th,
th { background: #0f1117 !important; color: #00d4ff !important; font-size: 13px !important;
     padding: 8px 12px !important; border: 1px solid #ffffff15 !important; }
.prose td, .md td, .markdown-body td,
td { color: #d0d0d0 !important; font-size: 13px !important;
     padding: 7px 12px !important; border: 1px solid #ffffff10 !important; }
.prose tr:nth-child(even), .md tr:nth-child(even), .markdown-body tr:nth-child(even),
tr:nth-child(even) { background: #161920 !important; }

/* ── Inline code ── */
.prose code, .md code, .markdown-body code,
code { background: #0f1117 !important; color: #10b981 !important;
       border-radius: 4px; padding: 2px 6px; border: 1px solid #ffffff12 !important; }

/* ── Labels ── */
label span, .label-wrap span, .block > label > span {
    color: #aaaaaa !important; font-size: 12px !important; font-weight: 600 !important;
}

/* ── Inputs & textareas ── */
textarea, input[type="text"] {
    background: #0f1117 !important; color: #e0e0e0 !important;
    border: 1px solid #ffffff18 !important; border-radius: 8px !important;
}

/* ── Analyze button ── */
button.primary, button.lg {
    background: linear-gradient(135deg, #00c4ee 0%, #7c3aed 100%) !important;
    border: none !important; border-radius: 10px !important;
    font-weight: 700 !important; font-size: 15px !important; color: #fff !important;
}

/* ── File upload area ── */
.upload-container, .file-preview {
    background: #0f1117 !important; border: 1.5px dashed #ffffff20 !important;
    border-radius: 10px !important; color: #aaa !important;
}

/* ── Suggestion list items ── */
.prose ol li, .md ol li, .prose ul li, .md ul li,
.markdown-body ol li, .markdown-body ul li {
    color: #d0d0d0 !important; font-size: 14px !important; line-height: 1.7 !important;
}
strong, b { color: #ffffff !important; }
em, i { color: #bbbbbb !important; }
"""

header_html = """
<div style="background: #1a1d27; border: 1px solid #ffffff0d; border-radius: 14px;
            padding: 18px 24px 16px; margin-bottom: 8px; text-align: center;">
    <h1 style="color: #fff; font-size: 22px; font-weight: 800; margin: 0 0 4px; letter-spacing: -0.5px;">
        🧠 Resume <span style="color:#00d4ff;">Intelligence</span> Platform
    </h1>
    <p style="color: #555; font-size: 13px; margin: 0 0 12px;">
        ML-Powered Resume ↔ Job Description Matching Engine
    </p>
    <div style="display: flex; justify-content: center; gap: 8px; flex-wrap: wrap;">
        <span style="background:#00d4ff12; color:#00d4ff; border:1px solid #00d4ff25; padding:3px 12px; border-radius:50px; font-size:11px; font-weight:700;">TF-IDF + ML</span>
        <span style="background:#7c3aed12; color:#a78bfa; border:1px solid #7c3aed25; padding:3px 12px; border-radius:50px; font-size:11px; font-weight:700;">Skill Gap Analysis</span>
    </div>
</div>
"""

with gr.Blocks(css=custom_css, title='Resume Intelligence Platform', theme=gr.themes.Base()) as app:
    gr.HTML(header_html)

    with gr.Tabs():

        # ── TAB 1: INPUT ──
        with gr.TabItem('📥 Input'):
            gr.Markdown('### Upload your resume and paste the job description below.')
            with gr.Row():
                with gr.Column(scale=1):
                    resume_input = gr.File(
                        label='📄 Upload Resume (PDF or DOCX)',
                        file_types=['.pdf', '.docx'],
                        type='filepath',
                    )
                    status_box = gr.Markdown(value='*Upload a resume to begin.*')
                with gr.Column(scale=1):
                    jd_input = gr.Textbox(
                        label='📋 Job Description',
                        placeholder='Paste the full job description here...\n\nExample:\nWe are looking for a Data Scientist with 2+ years of experience in Python, machine learning, TensorFlow, and SQL...',
                        lines=16, max_lines=30,
                    )
            analyze_btn = gr.Button('🚀  Analyze Resume Match', variant='primary', size='lg')

        # ── TAB 2: RESULTS ──
        with gr.TabItem('📊 Results'):
            with gr.Row():
                with gr.Column(scale=1):
                    gauge_output = gr.Plot(label='Match Score Gauge')
                    score_output = gr.Markdown(label='Score Breakdown')
                with gr.Column(scale=1):
                    model_output = gr.Markdown(label='Model Intelligence')
                    skills_output = gr.Markdown(label='Skill Analysis')
            sugg_output = gr.Markdown(label='Improvement Suggestions')

        # ── TAB 3: MODEL INSIGHTS ──
        with gr.TabItem('🔬 Model Insights'):
            cv_table_output = gr.Markdown(label='Cross-Validation Results')
            with gr.Row():
                f1_chart_output = gr.Plot(label='F1 Score Comparison')
                cm_chart_output = gr.Plot(label='Confusion Matrix')

    analyze_btn.click(
        fn=process,
        inputs=[resume_input, jd_input],
        outputs=[status_box, score_output, model_output, skills_output,
                 sugg_output, cv_table_output, gauge_output, f1_chart_output, cm_chart_output],
    )

app.launch(debug=True, share=True)
print('\n✅ Gradio app launched! Click the public URL above to open.')

---
## 🎓 Viva Preparation — Key Concepts

### Q1: Why TF-IDF and not raw counts?
> TF-IDF penalizes words that appear in many documents (like "the", "and") and rewards domain-specific terms. This makes it ideal for distinguishing resumes from different roles.

### Q2: Why StratifiedKFold instead of regular KFold?
> Our dataset may have class imbalance. StratifiedKFold ensures each fold has the same label ratio, giving more reliable F1 estimates.

### Q3: Why combine ML score + cosine similarity?
> The ML model captures latent patterns (e.g., which skill combinations lead to a match), while cosine similarity captures direct lexical overlap. Together they're more robust than either alone.

### Q4: Why CalibratedClassifierCV for SVM?
> LinearSVC doesn't output probabilities natively. We wrap it in CalibratedClassifierCV to get calibrated probability estimates needed for predict_proba().

### Q5: How does the synonym mapping work?
> We maintain a SYNONYM_MAP dict where each alias maps to a canonical skill name. E.g., 'ml' → 'machine learning', 'k8s' → 'kubernetes'. This allows skill extraction to work across different writing styles.

### Q6: What's the scoring formula?
> `Final Score = 0.6 × model_predict_proba + 0.4 × cosine_similarity`  
> The 0.6/0.4 weight gives more trust to the trained model while using cosine as a calibration anchor.

### Q7: How is the dataset labeled?
> Smart pairing: same-role resume+JD = 1 (match), unrelated resume vs tech JD = 0, cross-role mismatches = 0, DS vs MLE = 1 (borderline). This creates a realistic distribution.

---
*Resume Intelligence Platform | Built with Python, Gradio, scikit-learn*